In [4]:
import json

import requests
import time
import argparse


In [7]:


def create_session(domain, app_token, username, password):
    session = requests.Session()
    session.auth = (username, password)
    session.headers.update({
        "X-App-Token": app_token
    })
    return session


def create_job(session, domain, dataset_id):
    url = f"https://{domain}/api/imports2"

    payload = {
        "action": "replace",
        "viewUid": dataset_id
    }

    r = session.post(url, json=payload)
    r.raise_for_status()

    job_id = r.json()["id"]
    print(f"Created job: {job_id}")

    return job_id


def upload_file(session, domain, job_id, file_path):
    url = f"https://{domain}/api/imports2/{job_id}/files"

    with open(file_path, "rb") as f:
        r = session.post(url, files={"file": f})

    r.raise_for_status()
    print("File uploaded")


def run_job(session, domain, job_id):
    url = f"https://{domain}/api/imports2/{job_id}/run"

    r = session.post(url)
    r.raise_for_status()
    print("Job started")


def wait_for_job(session, domain, job_id):
    url = f"https://{domain}/api/imports2/{job_id}"

    while True:
        r = session.get(url)
        r.raise_for_status()

        status = r.json()
        state = status.get("state")

        print("Status:", state)

        if state in ["successful", "failed"]:
            return status

        time.sleep(5)


def run_replace(domain, app_token, username, password, dataset_id, file_path):
    session = create_session(domain, app_token, username, password)

    job_id = create_job(session, domain, dataset_id)
    upload_file(session, domain, job_id, file_path)
    run_job(session, domain, job_id)

    status = wait_for_job(session, domain, job_id)

    print("\nFinal Status:", status.get("state"))

    if status.get("state") == "failed":
        print("Error:", status.get("error"))
        print("Log:", status.get("log"))


if __name__ == "__main__":
    # parser = argparse.ArgumentParser()

    # parser.add_argument("--domain", required=True)
    # parser.add_argument("--app_token", required=True)
    # parser.add_argument("--username", required=True)
    # parser.add_argument("--password", required=True)
    # parser.add_argument("--dataset", required=True)
    # parser.add_argument("--file", required=True)
    print("start")
    flog=open(f"/home/joe/bic_etl/general/datasync/config.json")
    info = json.load(flog)
    username=info['username']
    password=info['password']
    api=info['appToken']
    domain = 'data.colorado.gov'
    # args = parser.parse_args()
    w4x4 = 'rvak-43ap'
    file = "persons_sol_ntcs.tsv"
    print(info)
    run_replace(
        domain,
        api,
        username,
        password,
        w4x4,
        file
    )

start
{'domain': 'https://data.colorado.gov', 'username': 'bic-help@xentity.com', 'password': 'FGjFVCu3L4pAdwfu6Gx', 'appToken': 'D1lR5IsRlplfS1K5QsfNSQ9cM', 'adminEmail': '', 'emailUponError': 'false', 'logDatasetID': '', 'outgoingMailServer': '', 'smtpPort': '', 'sslPort': '', 'smtpUsername': '', 'smtpPassword': ''}


HTTPError: 400 Client Error: Bad Request for url: https://data.colorado.gov/api/imports2